In [ ]:
from pathlib import Path
import os
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app.py").is_file())
os.chdir(PROJECT_DIR)
Path("reports/figures").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)


In [1]:
import pandas as pd

In [2]:
# Load dataset BISINDO Alphabet
df_bisindo = pd.read_csv(r'data/raw/BISINDO/Alphabet/landmarks_train.csv')

print("Dimensi Data BISINDO (Baris, Kolom):", df_bisindo.shape)
print("\nNama 10 Kolom Pertama:")
print(df_bisindo.columns[:10].tolist())

print("\nContoh 3 Baris Pertama:")
display(df_bisindo.head(3))

Dimensi Data BISINDO (Baris, Kolom): (7558, 127)

Nama 10 Kolom Pertama:
['h0_lm0_x', 'h0_lm0_y', 'h0_lm0_z', 'h0_lm1_x', 'h0_lm1_y', 'h0_lm1_z', 'h0_lm2_x', 'h0_lm2_y', 'h0_lm2_z', 'h0_lm3_x']

Contoh 3 Baris Pertama:


,h0_lm0_x,h0_lm0_y,h0_lm0_z,h0_lm1_x,h0_lm1_y,h0_lm1_z,h0_lm2_x,h0_lm2_y,h0_lm2_z,h0_lm3_x,...,h1_lm18_x,h1_lm18_y,h1_lm18_z,h1_lm19_x,h1_lm19_y,h1_lm19_z,h1_lm20_x,h1_lm20_y,h1_lm20_z,label
0,0.844895,0.759371,2.549733e-07,0.740274,0.761288,-0.083636,0.653328,0.736704,-0.136683,0.570540,...,0.180065,0.626359,-0.132338,0.195492,0.701841,-0.130561,0.206180,0.757120,-0.111790,0
1,0.870619,0.767059,-2.830261e-07,0.785775,0.749030,-0.052022,0.678575,0.702702,-0.094550,0.593281,...,0.100887,0.497598,-0.156935,0.083551,0.569969,-0.153424,0.088998,0.608152,-0.139035,0
2,0.201393,0.851281,6.282243e-07,0.302880,0.830268,-0.102970,0.378164,0.776020,-0.145549,0.460924,...,0.830046,0.539170,-0.095609,0.830206,0.627284,-0.101245,0.831318,0.691935,-0.087180,0


In [3]:
# Cek jumlah kelas unik dan daftar label
unique_labels = sorted(df_bisindo['label'].unique())
print(f"Total kelas unik: {len(unique_labels)}")
print(f"Daftar label: {unique_labels}")
print("\nSebaran jumlah data per label:")
print(df_bisindo['label'].value_counts().sort_index())

Total kelas unik: 26
Daftar label: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]

Sebaran jumlah data per label:
label
0     209
1     286
2     269
3     299
4     300
5     268
6     281
7     267
8     337
9     325
10    260
11    302
12    355
13    350
14    300
15    351
16    256
17    320
18    305
19    267
20    290
21    333
22    271
23    291
24    198
25    268
Name: count, dtype: int64


In [4]:
import os
import mediapipe as mp
import pandas as pd
from tqdm import tqdm
import cv2

# 1. Inisialisasi MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,       # Mode citra statis
    max_num_hands=1,              # SIBI didesain 1 tangan
    min_detection_confidence=0.5
)

# 2. Path folder SIBI V02 di direktori Anda
# Sesuaikan path jika letaknya berbeda
BASE_SIBI_DIR = os.path.join(
    "data/raw",
    "SIBI_datasets_LEMLITBANG_SIBI_R_90.10_V02", 
    "SIBI_datasets_LEMLITBANG_SIBI_R_90.10_V02"
)

# Kumpulkan dari training dan validation agar dataset SIBI kita melimpah
folders_to_process = ["training", "validation"]

data_rows = []
skipped_count = 0

print("Memulai ekstraksi koordinat MediaPipe untuk SIBI V02...")

for folder in folders_to_process:
    folder_path = os.path.join(BASE_SIBI_DIR, folder)
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        continue
        
    classes = sorted([d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))])
    
    for label in classes:
        class_dir = os.path.join(folder_path, label)
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        for img_name in tqdm(image_files, desc=f"{folder} - Huruf {label}", leave=False):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            
            if img is None:
                continue
                
            # Konversi BGR ke RGB untuk MediaPipe
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = hands.process(img_rgb)
            
            # Jika tangan terdeteksi, ambil 21 titik koordinat (x, y, z)
            if results.multi_hand_landmarks:
                hand_lms = results.multi_hand_landmarks[0]
                row = []
                for lm in hand_lms.landmark:
                    row.extend([lm.x, lm.y, lm.z])
                row.append(label)  # Label huruf (A, B, C, ...)
                data_rows.append(row)
            else:
                skipped_count += 1

hands.close()

# 3. Bentuk Header Kolom: h0_lm0_x s.d. h0_lm20_z + label
columns = []
for i in range(21):
    columns.extend([f"h0_lm{i}_x", f"h0_lm{i}_y", f"h0_lm{i}_z"])
columns.append("label")

# 4. Simpan ke CSV
df_sibi = pd.DataFrame(data_rows, columns=columns)
df_sibi.to_csv("data/processed/sibi_landmarks.csv", index=False)

print("\n" + "="*50)
print(f"Ekstraksi SIBI Selesai!")
print(f"Total baris koordinat berhasil diekstrak : {len(df_sibi)}")
print(f"Gambar yang gagal terdeteksi (dilewati)  : {skipped_count}")
print(f"Dimensi data SIBI: {df_sibi.shape}")
print(f"File tersimpan sebagai: sibi_landmarks.csv")
print("="*50)

df_sibi.head(3)

Memulai ekstraksi koordinat MediaPipe untuk SIBI V02...
Folder tidak ditemukan: SIBI_datasets_LEMLITBANG_SIBI_R_90.10_V02\SIBI_datasets_LEMLITBANG_SIBI_R_90.10_V02\training
Folder tidak ditemukan: SIBI_datasets_LEMLITBANG_SIBI_R_90.10_V02\SIBI_datasets_LEMLITBANG_SIBI_R_90.10_V02\validation

Ekstraksi SIBI Selesai!
Total baris koordinat berhasil diekstrak : 0
Gambar yang gagal terdeteksi (dilewati)  : 0
Dimensi data SIBI: (0, 64)
File tersimpan sebagai: sibi_landmarks.csv


,h0_lm0_x,h0_lm0_y,h0_lm0_z,h0_lm1_x,h0_lm1_y,h0_lm1_z,h0_lm2_x,h0_lm2_y,h0_lm2_z,h0_lm3_x,...,h0_lm18_x,h0_lm18_y,h0_lm18_z,h0_lm19_x,h0_lm19_y,h0_lm19_z,h0_lm20_x,h0_lm20_y,h0_lm20_z,label
